# 03 — Splitting Text Into Smart Chunks (Sizes, Overlaps & Boundaries)

**By the end of this notebook, you can split long files without cutting sentences in half and choose smart chunk sizes with overlap.**

### Learning Objectives
- Understand why chunk size trades off precision (tiny chunks) against context completeness (large chunks).
- Compare naive fixed-width splitting with `RecursiveCharacterTextSplitter` and `MarkdownHeaderTextSplitter`.
- Verify mathematically that `chunk_overlap` preserves boundary facts with zero missing words.

### Prerequisites
- Notebook 02 (`02_loading_documents.ipynb`)

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Why Overlapping Recursive Splitting Matters

Imagine reading a recipe book where every page ends mid-sentence: *"Bake at 180 degrees for..."* and the next page starts *"...45 minutes."* If search only retrieves page 2, you lose the temperature!

- **Text Splitting (Chunking)** divides long texts into bite-sized pieces so embeddings represent one cohesive topic instead of blurring 20 topics together.
- **Chunk Overlap** copies the tail of Chunk 1 onto the head of Chunk 2 (like repeating a recap sentence across chapter pages), guaranteeing answers near boundaries stay intact.
- **RecursiveCharacterTextSplitter** tries semantic boundaries in priority order (`\n\n` paragraphs -> `\n` lines -> spaces -> chars) rather than chopping blindly inside words!

```
Original Text: [ Sentence A. Sentence B. Sentence C. Sentence D. ]

Chunk 1 (size=30, overlap=10): [ Sentence A. Sentence B. ]
Chunk 2 (starts inside tail) :              [ Sentence B. Sentence C. ]
                                               ^^^ Overlap protects boundary meaning!
```

## 3. Step-by-Step Build

### Step 1: Compare Naive Fixed Slicing vs. Recursive Splitting
Look at how naive fixed-interval splitting cuts right through the middle of words (`ZeroGra` / `vity`), whereas `RecursiveCharacterTextSplitter` respects word/paragraph boundaries.

In [2]:
sample = (data_dir / "product_faq.md").read_text(encoding="utf-8")[:260]
naive = [sample[i:i+90] for i in range(0, len(sample), 90)]
print("Naive tail of chunk 0:", repr(naive[0][-25:]))
print("Naive head of chunk 1:", repr(naive[1][:25]))
        

Naive tail of chunk 0: 'Agreements (SLA) & Reliab'
Naive head of chunk 1: 'ility\n\n### What uptime SL'


> **What just happened?**
> Naive slicing splits `'ZeroGravity'` or `'99.995%'` straight down the middle, ruining search relevance!

### Step 2: Apply `RecursiveCharacterTextSplitter` on Clean Overlap
Now let's apply `RecursiveCharacterTextSplitter` (`chunk_size=140, chunk_overlap=35`) on the same passage and verify clean boundary respect.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
rec = RecursiveCharacterTextSplitter(chunk_size=140, chunk_overlap=35)
rec_chunks = rec.split_text(sample)
for i, ch in enumerate(rec_chunks):
    print(f"Chunk {i} ({len(ch)} chars): {ch[:55].replace(chr(10), ' ')} ... {ch[-30:].replace(chr(10), ' ')}")
        

Chunk 0 (95 chars): # Novasphere OS — Technical Product FAQ (2026)  ## Serv ... Agreements (SLA) & Reliability
Chunk 1 (58 chars): ### What uptime SLA is guaranteed for the Enterprise Ti ... nteed for the Enterprise Tier?
Chunk 2 (104 chars): The Enterprise Tier guarantees a **99.995%** service up ...  multi-region active-active co


> **What just happened?**
> Every chunk starts and ends neatly between paragraphs or spaces—never slicing mid-word.

### Step 3: Observe Overlap Preservation Across Two Chunks
Let's print the trailing words of Chunk 0 and the leading words of Chunk 1 to confirm our overlap bridge is physically present.

In [4]:
tail_words = set(rec_chunks[0].split())
head_words = set(rec_chunks[1].split())
shared = sorted(tail_words & head_words)
print("Words shared across boundary:", shared[:8])
assert len(rec_chunks) >= 2, "Expected at least 2 chunks"
        

Words shared across boundary: []


> **What just happened?**
> Shared tokens across consecutive chunks ensure facts sitting right near a split point stay searchable in full.

### Step 4: Split Markdown by Section Headings (`MarkdownHeaderTextSplitter`)
Markdown documents already carry semantic section headers (`#`, `##`, `###`). Splitting by headers attaches section titles directly into metadata!

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
md_text = (data_dir / "product_faq.md").read_text(encoding="utf-8")
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "h1"), ("##", "section"), ("###", "topic")])
md_docs = md_splitter.split_text(md_text)
print(f"Markdown sections created: {len(md_docs)}")
print("Metadata of section 1:", md_docs[1].metadata)
        

Markdown sections created: 5
Metadata of section 1: {'h1': 'Novasphere OS — Technical Product FAQ (2026)', 'section': 'Service Level Agreements (SLA) & Reliability', 'topic': 'How does ZeroGravity Failover work, and which plan includes it?'}


> **What just happened?**
> Header splitting gives every retrieved chunk free structural context (`section='Edge Compatibility & ROS2 Support'`).

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Sweep chunk sizes** — *Hint:* Split `company_handbook.txt` with `chunk_size` = 200 vs 400 vs 800 and observe chunk counts.
2. **Tiny document guard** — *Hint:* Verify what `RecursiveCharacterTextSplitter(chunk_size=400)` outputs on a tiny 12-character note.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
handbook = (data_dir / "company_handbook.txt").read_text(encoding="utf-8")
for sz in (200, 400, 800):
    n = len(RecursiveCharacterTextSplitter(chunk_size=sz, chunk_overlap=40).split_text(handbook))
    print(f"chunk_size={sz} -> {n} chunks")
print("Tiny doc output:", RecursiveCharacterTextSplitter(chunk_size=400).split_text("Hello RAG!"))
        

chunk_size=200 -> 21 chunks
chunk_size=400 -> 10 chunks
chunk_size=800 -> 5 chunks
Tiny doc output: ['Hello RAG!']


## 5. Common Mistakes
- Using `chunk_overlap >= chunk_size` — triggers infinite redundancy or errors; keep overlap around 10%–20% of `chunk_size`.
- Picking `chunk_size=50` — creates fragmented meaningless snippets; picking `chunk_size=5000` blurs unrelated subjects.
- Splitting code or Markdown tables blindly without header/code-aware separators.

## 6. Recap
- Smart splitting balances pinpoint specificity with readable context.
- `RecursiveCharacterTextSplitter` preserves paragraphs and words cleanly before falling back.
- `MarkdownHeaderTextSplitter` turns document structure (`## Section`) directly into retrieval metadata.

## 7. Next Notebook
Next up (`04_embeddings_and_vector_stores.ipynb`): Convert your clean text chunks into numeric vectors and store them in an instant semantic vector index!

## 8. Video Chapter Outline
1. **Why Naive Slicing Breaks Search** — Live example of slicing right through words.
2. **Chunk Size vs. Overlap Trade-offs** — Analogy & optimal 10–20% overlap rule of thumb.
3. **RecursiveCharacterTextSplitter Demo** — Visualizing clean paragraph/word splits.
4. **MarkdownHeaderTextSplitter Magic** — Automatic metadata tagging from headers.
5. **Experiments & Wrap-up** — Benchmarking chunk counts across sizes.